# 04 — InceptionV3 + Classical ML Ensemble

Uses the InceptionV3 model from `03_InceptionV3.ipynb` as a feature extractor. Input size: 299×299.

## Section 0: Google Colab Setup & Dataset Download

**Run this cell first every time you open a new Colab session.**

### One-time Colab Secrets setup
Go to **Runtime → Manage secrets** and add these three secrets:

| Secret name | Value |
|---|---|
| `KAGGLE_USERNAME` | sk1285 |
| `KAGGLE_KEY` | (your kaggle API key from kaggle.com/settings/account) |
| `HF_TOKEN` | (your HuggingFace token from huggingface.co/settings/tokens) |

Once secrets are saved they persist across all Colab sessions — you only do this once.

### What this cell does
- Installs `kaggle` and `huggingface_hub`
- Reads credentials from Colab Secrets
- Downloads the Brain Tumor MRI dataset once to `/content/MRI_DATASET/`  
  (all 8 notebooks share the same folder — subsequent notebooks skip the download)
- Sets path variables used by later cells

In [ ]:
import sys, os, json

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Google Colab detected — setting up environment...")
    os.system("pip install kaggle huggingface_hub -q")

    # Read credentials from Colab Secrets (Runtime → Manage secrets)
    from google.colab import userdata
    _kaggle_user = userdata.get('KAGGLE_USERNAME')   # secret name: KAGGLE_USERNAME
    _kaggle_key  = userdata.get('KAGGLE_KEY')         # secret name: KAGGLE_KEY
    HF_TOKEN     = userdata.get('HF_TOKEN')            # secret name: HF_TOKEN

    # Write kaggle.json so the kaggle CLI can authenticate
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as _f:
        json.dump({'username': _kaggle_user, 'key': _kaggle_key}, _f)
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

    # Download dataset once — shared across all 8 notebooks
    DATASET_PATH     = "/content/MRI_DATASET/"
    SAVED_MODELS_DIR = "/content/saved_models/"
    RESULTS_DIR      = "/content/results/"

    if not os.path.exists(DATASET_PATH + "Training"):
        print("Downloading Brain Tumor MRI dataset (≈ 150 MB)...")
        os.system("kaggle datasets download masoudnickparvar/brain-tumor-mri-dataset -p /content/")
        os.system(f"unzip -q /content/brain-tumor-mri-dataset.zip -d {DATASET_PATH}")
        os.system("rm -f /content/brain-tumor-mri-dataset.zip")
        print("✓ Dataset downloaded")
    else:
        print("✓ Dataset already present — skipping download")

else:
    print("Running locally")
    DATASET_PATH     = "../MRI_DATASET/"
    SAVED_MODELS_DIR = "../saved_models/"
    RESULTS_DIR      = "../results/"
    HF_TOKEN         = os.environ.get('HF_TOKEN', '')

os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"  Dataset path : {DATASET_PATH}")
print(f"  Models path  : {SAVED_MODELS_DIR}")
print(f"  Results path : {RESULTS_DIR}")
print("✓ Environment ready")

## Section 1: Imports & Configuration

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             roc_curve, auc, precision_recall_curve,
                             average_precision_score)
from sklearn.preprocessing import label_binarize
import joblib
print("✓ Imports complete")

## Section 2: Constants & Hyperparameters

In [ ]:
NOTEBOOK_NAME    = "04_InceptionV3_Ensemble"

DATASET_PATH     = globals().get("DATASET_PATH", "../MRI_DATASET/")
TRAIN_DIR        = DATASET_PATH + "Training/"
TEST_DIR         = DATASET_PATH + "Testing/"

CLASS_NAMES      = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES      = 4

IMG_HEIGHT       = 299  # InceptionV3/Xception requires 299×299
IMG_WIDTH        = 299
CHANNELS         = 3

BATCH_SIZE       = 32
LEARNING_RATE    = 1e-4
VALIDATION_SPLIT = 0.2
RANDOM_SEED      = 42

SAVED_MODELS_DIR = globals().get("SAVED_MODELS_DIR", "../saved_models/")
RESULTS_DIR      = globals().get("RESULTS_DIR", "../results/")
RESULTS_NB_DIR   = os.path.join(RESULTS_DIR, NOTEBOOK_NAME)
HF_TOKEN         = globals().get("HF_TOKEN", os.environ.get("HF_TOKEN", ""))
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_NB_DIR, exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)

print("✓ Constants configured")
print(f"  Image size : {IMG_HEIGHT}×{IMG_WIDTH}")
print(f"  Classes    : {CLASS_NAMES}")

## Section 3: Data Loading & Verification

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=VALIDATION_SPLIT
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    subset='training',
    seed=RANDOM_SEED,
    shuffle=True
)
val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    subset='validation',
    seed=RANDOM_SEED,
    shuffle=False
)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)

# No-augmentation generator for feature extraction (deterministic output)
feature_datagen = ImageDataGenerator(rescale=1./255)
feature_train_generator = feature_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)

print("=" * 50)
print("DATA VERIFICATION")
print(f"Class indices      : {train_generator.class_indices}")
print(f"Training samples   : {train_generator.samples}")
print(f"Validation samples : {val_generator.samples}")
print(f"Test samples       : {test_generator.samples}")
print(f"Image size         : {IMG_HEIGHT}×{IMG_WIDTH}")
print(f"Batch size         : {BATCH_SIZE}")
print("=" * 50)

## Section 4: Data Preprocessing & Augmentation

In [ ]:
# Preprocessing: rescale 1/255 via ImageDataGenerator.
print("✓ Preprocessing configured via ImageDataGenerator (rescale 1/255)")

## Section 5: Model Definition

In [ ]:
inc_model_path = SAVED_MODELS_DIR + 'inceptionv3_model.h5'
inc_base = load_model(inc_model_path)
print(f"✓ InceptionV3 model loaded from: {inc_model_path}")

feature_extractor = Model(
    inputs=inc_base.input,
    outputs=inc_base.get_layer('feature_layer').output,
    name='inceptionv3_feature_extractor'
)
print(f"✓ Feature extractor — output shape: {feature_extractor.output_shape}")

rf  = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
dt  = DecisionTreeClassifier(random_state=RANDOM_SEED)
svm = SVC(probability=True, random_state=RANDOM_SEED, kernel='rbf', C=1.0)
ensemble_clf = VotingClassifier(
    estimators=[('rf', rf), ('dt', dt), ('svm', svm)],
    voting='soft'
)
print("✓ Classifiers defined: RF, DT, SVM, Soft-Vote Ensemble")

## Section 6: Model Training

In [ ]:
print("Extracting training features...")
feature_train_generator.reset()
X_train = feature_extractor.predict(feature_train_generator, verbose=1)
y_train = feature_train_generator.classes
print(f"✓ Training features: {X_train.shape}")

print("Extracting test features...")
test_generator.reset()
X_test = feature_extractor.predict(test_generator, verbose=1)
y_test  = test_generator.classes
print(f"✓ Test features: {X_test.shape}")

print("\nTraining Random Forest...")
rf.fit(X_train, y_train)
print("✓ Random Forest trained")

print("Training Decision Tree...")
dt.fit(X_train, y_train)
print("✓ Decision Tree trained")

print("Training SVM...")
svm.fit(X_train, y_train)
print("✓ SVM trained")

print("Training Soft-Vote Ensemble...")
ensemble_clf.fit(X_train, y_train)
print("✓ Ensemble trained")

## Section 7: Model Evaluation

In [ ]:
import re as _re

def evaluate_sklearn_model(clf, X_test, y_test, model_name="Model"):
    """Standard evaluation for sklearn classifiers."""
    _fname = _re.sub(r"[^a-z0-9]+", "_", model_name.lower()).strip("_")
    y_pred       = clf.predict(X_test)
    y_pred_proba = clf.predict_proba(X_test)
    acc = accuracy_score(y_test, y_pred)

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'{model_name} — Confusion Matrix (Acc: {acc:.4f})')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_NB_DIR, f"{_fname}_confusion_matrix.jpg"),
                dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\n{model_name} — Classification Report")
    print("=" * 60)
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

    # ROC Curve
    y_true_bin = label_binarize(y_test, classes=[0, 1, 2, 3])
    plt.figure(figsize=(8, 6))
    for i, cls in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{cls} (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} — ROC Curve')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_NB_DIR, f"{_fname}_roc_curve.jpg"),
                dpi=150, bbox_inches='tight')
    plt.show()

    # Precision-Recall Curve
    plt.figure(figsize=(8, 6))
    for i, cls in enumerate(CLASS_NAMES):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_proba[:, i])
        ap = average_precision_score(y_true_bin[:, i], y_pred_proba[:, i])
        plt.plot(recall, precision, label=f'{cls} (AP = {ap:.2f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'{model_name} — Precision-Recall Curve')
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_NB_DIR, f"{_fname}_pr_curve.jpg"),
                dpi=150, bbox_inches='tight')
    plt.show()

    return acc, y_pred, y_pred_proba

rf_acc, rf_pred, rf_proba = evaluate_sklearn_model(rf, X_test, y_test, "InceptionV3 + Random Forest")
dt_acc, dt_pred, dt_proba = evaluate_sklearn_model(dt, X_test, y_test, "InceptionV3 + Decision Tree")
svm_acc, svm_pred, svm_proba = evaluate_sklearn_model(svm, X_test, y_test, "InceptionV3 + SVM")
ensemble_acc, ensemble_pred, ensemble_proba = evaluate_sklearn_model(ensemble_clf, X_test, y_test, "InceptionV3 + Soft-Vote Ensemble")

print("\n" + "=" * 50)
print("ACCURACY COMPARISON")
print(f"  InceptionV3 + Random Forest  : {rf_acc:.4f}")
print(f"  InceptionV3 + Decision Tree  : {dt_acc:.4f}")
print(f"  InceptionV3 + SVM            : {svm_acc:.4f}")
print(f"  InceptionV3 + Ensemble       : {ens_acc:.4f}")
print("=" * 50)

## Section 8: Save Model

In [ ]:
joblib.dump(rf,  SAVED_MODELS_DIR + 'inceptionv3_rf.pkl')
joblib.dump(dt,  SAVED_MODELS_DIR + 'inceptionv3_dt.pkl')
joblib.dump(svm, SAVED_MODELS_DIR + 'inceptionv3_svm.pkl')
joblib.dump(ensemble_clf, SAVED_MODELS_DIR + 'inceptionv3_ensemble_model.pkl')
print(f"✓ Models saved to {SAVED_MODELS_DIR}")
print("  inceptionv3_rf.pkl, inceptionv3_dt.pkl, inceptionv3_svm.pkl, inceptionv3_ensemble_model.pkl")

## Section 9: Results Summary

In [ ]:
print("=" * 60)
print(f"NOTEBOOK: {NOTEBOOK_NAME}")
print(f"Dataset  : {train_generator.samples + val_generator.samples} training images")
print(f"Classes  : {CLASS_NAMES}")
print(f"Image size: {IMG_HEIGHT}×{IMG_WIDTH}")
print(f"Batch size: {BATCH_SIZE}, Seed: {RANDOM_SEED}")
print(f"Feature dim: {X_train.shape[1]} (Dense layer output)")
print("-" * 60)
print(f"  InceptionV3 + Random Forest  : {rf_acc:.4f}")
print(f"  InceptionV3 + Decision Tree  : {dt_acc:.4f}")
print(f"  InceptionV3 + SVM            : {svm_acc:.4f}")
print(f"  InceptionV3 + Ensemble       : {ens_acc:.4f}")
print("=" * 60)
print("Saved models location:", SAVED_MODELS_DIR)

## Section 10: HuggingFace Upload

Uploads the model file(s) saved in Section 8 and all result JPGs from Section 7 to `shehank98/brain-tumor-mri-models` on HuggingFace Hub.

Requires `HF_TOKEN` to be set (via Colab Secrets in Section 0, or `HF_TOKEN` env var).

In [ ]:
# ── HuggingFace repository ────────────────────────────────────────────────────
HF_REPO_ID = "shehank98/brain-tumor-mri-models"   # your HF repo

try:
    from huggingface_hub import HfApi, login as hf_login
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'huggingface_hub', '-q'])
    from huggingface_hub import HfApi, login as hf_login

if not HF_TOKEN:
    print("WARNING: HF_TOKEN not set — skipping HuggingFace upload.")
    print("  Set it in Colab Secrets (key: HF_TOKEN) or as an env var.")
else:
    hf_login(token=HF_TOKEN, add_to_git_credential=False)
    api = HfApi()

    # Create repo if it does not exist yet
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model",
                    private=False, exist_ok=True)
    print(f"✓ Repository ready: https://huggingface.co/{HF_REPO_ID}")

    # Upload model files
    _model_files = ['inceptionv3_rf.pkl', 'inceptionv3_dt.pkl', 'inceptionv3_svm.pkl', 'inceptionv3_ensemble_model.pkl']
    for _fname in _model_files:
        _local = os.path.join(SAVED_MODELS_DIR, _fname)
        if not os.path.exists(_local):
            print(f"  SKIP (not found): {_fname}")
            continue
        _size = os.path.getsize(_local) / 1e6
        print(f"  Uploading {_fname} ({_size:.1f} MB)...", end="", flush=True)
        api.upload_file(
            path_or_fileobj=_local,
            path_in_repo=f"models/{_fname}",
            repo_id=HF_REPO_ID,
            commit_message=f"Upload {_fname} from 04_InceptionV3_Ensemble",
        )
        print(" done")

    # Upload results JPGs
    _results_nb = os.path.join(RESULTS_DIR, NOTEBOOK_NAME)
    if os.path.exists(_results_nb):
        _jpgs = [f for f in os.listdir(_results_nb) if f.endswith('.jpg')]
        for _jpg in sorted(_jpgs):
            print(f"  Uploading result chart {_jpg}...", end="", flush=True)
            api.upload_file(
                path_or_fileobj=os.path.join(_results_nb, _jpg),
                path_in_repo=f"results/{NOTEBOOK_NAME}/{_jpg}",
                repo_id=HF_REPO_ID,
                commit_message=f"Add result chart {_jpg}",
            )
            print(" done")
        print(f"✓ {len(_jpgs)} result charts uploaded")
    else:
        print("  No result charts found — run Section 7 first")

    print(f"\n✓ Upload complete: https://huggingface.co/{HF_REPO_ID}")